In [17]:
#Dependencies
import os #navigating directories
import torch #Main library for machine library functions
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import tqdm #Progress bar
from torch.utils.data import Dataset, DataLoader
import numpy as np #External math will be handled with numpy
import matplotlib #Plots generated will be from this library
from PIL import Image, ImageOps #Image processing along with consistent padding for data
import matplotlib.pyplot as plt # Add this line!

In [18]:
#These are our constants for the script
IMG_SIZE = 64 #Base 2, important for pooling keeps things even
BATCH_SIZE = 32 #images used in each dataset
EPOCHS = 10 #Iterations through the training dataset
LEARNING_RATE = 0.001 #Testing this, may have to change, ran at 86% accuracy, we're chilling

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #pytorch said use this
print(f"Using device: {device}")


In [19]:
class XRayDataset(Dataset):
    def __init__(self, dir, img_size=IMG_SIZE): #Should be ran with XRayDataset(training_dir)
        self.img_size = img_size
        self.image_paths = []
        self.labels = []

        # 0 = NORMAL, 1 = PNEUMONIA
        self.classes = ['NORMAL', 'PNEUMONIA']

        for label_idx, class_name in enumerate(self.classes):
            class_dir = os.path.join(dir, class_name)

            if not os.path.exists(class_dir):
                print(f"Warning: Missing folder - {class_dir}")
                continue

            for filename in os.listdir(class_dir):
                if filename.lower().endswith(('.jpeg', '.jpg', '.png')):
                    self.image_paths.append(os.path.join(class_dir, filename))
                    self.labels.append(label_idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        img = Image.open(img_path).convert('L')
        img = ImageOps.pad(img, (self.img_size, self.img_size), color=0)

        img_array = np.array(img, dtype=np.float32) / 255.0 #set values from [0,1]
        img_array = np.expand_dims(img_array, axis=0) #add extra dimension for pytorch tensor

        return torch.tensor(img_array), torch.tensor(label, dtype=torch.long)

In [20]:
"""
#First 3 lines are colab specific
from google.colab import drive
drive.mount('/content/drive')
root = "/content/drive/MyDrive/math156/chest_xray"
"""
root = 'dataset/chest_xray'

TRAIN_DIR = os.path.join(root, "train")
TEST_DIR = os.path.join(root, "test")

train_dataset = XRayDataset(TRAIN_DIR)
test_dataset = XRayDataset(TEST_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True) #pytorch function,
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Loaded {len(train_dataset)} training X-Rays and {len(test_dataset)} testing X-Rays.")

Loaded 5232 training X-Rays and 624 testing X-Rays.


In [21]:
class GrayScaleCNN(nn.Module):
    def __init__(self, num_classes):
        super(GrayScaleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # 64x64 shrinks in half 3 times -> 32 -> 16 -> 8
        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, num_classes) # Automatically sizes output to your folders

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 64 * 8 * 8) # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [22]:
#Main training section
if len(train_dataset) > 0:
    num_classes = len(train_dataset.classes)
    model = GrayScaleCNN(num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("\nStarting Training...")

history_loss = []

for epoch in range(EPOCHS): #loop through
    model.train()
    running_loss = 0.0

    loop = tqdm.tqdm(train_loader, desc=f'Epoch [{epoch+1}/{EPOCHS}]', leave=True) # progress bar, unimportant just adds visual

    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device) 

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    epoch_avg_loss = running_loss / len(train_loader) # finds average loss

    history_loss.append(epoch_avg_loss) # store loss for plotting

    loop.set_postfix(avg_loss=epoch_avg_loss)

NameError: name 'device' is not defined

In [ ]:
print("\nEvaluating on Test Data...")
model.eval()
correct = 0
total = 0

# Turn off gradients to save memory and speed up testing
with torch.no_grad():
  for inputs, labels in test_loader:
      inputs, labels = inputs.to(device), labels.to(device)
      outputs = model(inputs)

      _, predicted = torch.max(outputs.data, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()

if total > 0:
  print(f"Final Accuracy on Test Set: {100 * correct / total:.2f}%")

In [ ]:
print("\nGenerating Loss Graph...")

plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), history_loss, marker='o', linestyle='-', color='red', linewidth=2)
plt.title('Cross-Entropy Loss over Training Epochs', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Average Loss', fontsize=12)


plt.xticks(range(1, EPOCHS + 1))


plt.grid(True, linestyle='--', alpha=0.7)
plt.show()